# PyRestore 04 — Case B mechanics: paired-scene urban-renewal detection (synthetic)

A renewal claim needs temporal evidence: two comparable captures of the same site. This notebook
demonstrates the paired-scene **mechanics** end to end on clearly-labelled synthetic imagery —
pair derivation with a comparability audit, the expert-knowledge-guided renewal prompt, CV
change deltas, and validation with abstentions.

**Data status:** no real before/after panel ships with this repository (the local Shenzhen
collection contains different years but no same-location panel at audited tolerances). Real-data
routes and the labelling protocol: `docs/CASE_B_PROTOCOL.md`.

In [1]:
from pathlib import Path

import pandas as pd
from PIL import Image

from pyrestore import load_config, make_pairs, run_task

demo = Path("data/demo_pairs")
demo.mkdir(parents=True, exist_ok=True)

# Four synthetic sites x two epochs. Distinct palettes stand in for visual change.
rows = []
for site in range(4):
    for year, color in ((2017, (120, 90, 60)), (2022, (60, 140, 80))):
        capture_id = f"site{site}_{year}"
        path = demo / f"{capture_id}.png"
        Image.new("RGB", (96, 64), color).save(path)
        rows.append({"site_id": f"site{site}", "capture_id": capture_id, "image_path": str(path),
                     "lon": 2.20 + site * 0.01, "lat": 41.41 + site * 0.008,
                     "captured_at": f"{year}-06", "heading": 45})
manifest = pd.DataFrame(rows)
portable_manifest = manifest.copy()
portable_manifest["image_path"] = portable_manifest["image_path"].map(
    lambda value: Path(value).name
)
portable_manifest.to_csv(demo / "manifest.csv", index=False)
manifest.head()

,site_id,capture_id,image_path,lon,lat,captured_at,heading
0,site0,site0_2017,data/demo_pairs/site0_2017.png,2.20,41.410,2017-06,45
1,site0,site0_2022,data/demo_pairs/site0_2022.png,2.20,41.410,2022-06,45
2,site1,site1_2017,data/demo_pairs/site1_2017.png,2.21,41.418,2017-06,45
3,site1,site1_2022,data/demo_pairs/site1_2022.png,2.21,41.418,2022-06,45
4,site2,site2_2017,data/demo_pairs/site2_2017.png,2.22,41.426,2017-06,45


## 1. Pair derivation is an audited contract, not an assumption

Each candidate pair is checked against the task's `pair_contract` (distance, heading, known capture dates); failures stay in the table with a reason.

In [2]:
pairs = make_pairs(manifest, max_distance_m=20, max_heading_diff_deg=30, require_captured_at=True)
pairs[["pair_id", "site_id", "before_capture_id", "after_capture_id",
       "heading_diff_deg", "distance_m", "comparability_status", "exclusion_reason"]]

,pair_id,site_id,before_capture_id,after_capture_id,heading_diff_deg,distance_m,comparability_status,exclusion_reason
0,pair_40c763c683b4,site0,site0_2017,site0_2022,0.0,0.0,ok,
1,pair_cac279c1bbdd,site1,site1_2017,site1_2022,0.0,0.0,ok,
2,pair_0a5c6fcf2ad5,site2,site2_2017,site2_2022,0.0,0.0,ok,
3,pair_059c332e3060,site3,site3_2017,site3_2022,0.0,0.0,ok,


## 2. What breaks the audit: same-date and missing-date candidates

In [3]:
broken = manifest.copy()
broken.loc[broken.index[:2], "captured_at"] = None          # site0 loses its dates
same_date = manifest.copy()
same_date["captured_at"] = "2022-06"                         # nobody spans time

audit_broken = make_pairs(broken)
audit_same = make_pairs(same_date)
print(audit_broken[["site_id", "comparability_status", "exclusion_reason"]].to_string(index=False))
print(audit_same["exclusion_reason"].value_counts().to_dict())

site_id comparability_status    exclusion_reason
  site0             excluded missing_captured_at
  site1                   ok                    
  site2                   ok                    
  site3                   ok                    
{'same_capture_date': 4}


## 3. Run the renewal task (VLM reply injected; CV channel runs for real deltas)

In [4]:
import json
from types import SimpleNamespace


class InlineVLMClient:
    def __init__(self, reply): self.reply, self.calls = reply, 0
    def _create(self, **kwargs):
        self.calls += 1
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=self.reply))])
    @property
    def chat(self):
        return SimpleNamespace(completions=SimpleNamespace(create=self._create))

reply = json.dumps({
    "renewal_detected": "yes",
    "renewal_types": ["greening_landscape"],
    "nuisance": ["illumination"],
    "confidence": 0.7,
    "evidence": "vegetation share increases between the captures",
})
def demo_cv_extractor(image_path, cfg):
    """Deterministic CV stand-in for the synthetic images (green-dominance share).

    The live extractor (pyrestore.cv.extract_cv_batch) needs the optional torch extra; this
    notebook is a mechanics demonstration on synthetic images, so a transparent stand-in
    keeps it runnable everywhere. The 2022 palettes are greener -> positive GVI delta.
    """
    import numpy as np
    from PIL import Image

    arr = np.asarray(Image.open(image_path).convert("RGB"), dtype=float) / 255.0
    green_share = float(((arr[:, :, 1] > arr[:, :, 0]) & (arr[:, :, 1] > arr[:, :, 2])).mean())
    return {"seg_vegetation": green_share, "seg_sky": 0.2, "seg_building": 0.1,
            "seg_wall": 0.02, "seg_fence": 0.01, "GVI": green_share, "SVF": 0.2,
            "enclosure": 0.13}

cfg = load_config({"vlm": {"cache_db": "cache/demo_renewal_cache.sqlite"}})
result = run_task(demo / "manifest.csv", "tasks/renewal.yaml", cfg,
                  cv_extractor=demo_cv_extractor, vlm_client=InlineVLMClient(reply),
                  make_map=False)
result["indicators"][["pair_id", "before_capture_id", "after_capture_id",
                      "GVI_delta", "SVF_delta", "vlm_renewal_detected",
                      "vlm_confidence", "vlm_evidence"]]

,pair_id,before_capture_id,after_capture_id,GVI_delta,SVF_delta,vlm_renewal_detected,vlm_confidence,vlm_evidence
0,pair_40c763c683b4,site0_2017,site0_2022,1.0,0.0,yes,0.7,vegetation share increases between the captures
1,pair_cac279c1bbdd,site1_2017,site1_2022,1.0,0.0,yes,0.7,vegetation share increases between the captures
2,pair_0a5c6fcf2ad5,site2_2017,site2_2022,1.0,0.0,yes,0.7,vegetation share increases between the captures
3,pair_059c332e3060,site3_2017,site3_2022,1.0,0.0,yes,0.7,vegetation share increases between the captures


## 4. Validation with abstentions: `uncertain` is never hidden inside F1

In [5]:
from pyrestore.validate import validate_classification

labels = pd.DataFrame({
    "pair_id": result["indicators"]["pair_id"],
    # illustration labels matching the injected replies
    "renewal_label": ["yes"] * len(result["indicators"]),
})
report = validate_classification(
    result["indicators"], labels,
    label_column="renewal_label",
    prediction_column="vlm_renewal_detected",
    positive_value="yes",
    abstain_values=("uncertain",),
)
report

,kind,n,n_overlap,n_evaluable,n_missing_reference,n_missing_prediction,n_invalid_reference,n_invalid_prediction,reference_type,positive_value,...,tn,n_abstain,precision,recall,f1,accuracy,coverage,overall_recall,overall_accuracy,status
0,classification,4,4,4,0,0,0,0,unspecified_reference,yes,...,0,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,validated


## 5. From mechanics to a real case

1. Acquire a same-location panel (Mapillary sequences in Milan/Barcelona, or Baidu historical
   panoramas in Shenzhen) — routes, costs and licence notes in `docs/CASE_B_PROTOCOL.md`.
2. `pairs = make_pairs("real.csv"); pairs.to_csv("real_pairs.csv", index=False)` → audit coverage before labelling.
3. Human-label ≥150–300 pairs incl. ≥30% nuisance negatives (season / temporary use / maintenance).
4. Model pre-experiment on ~50 pairs, then run the full task with `reference=labels.csv`.

**Prohibited wording (terminology contract):** a single-image condition score is not renewal
detection; a VLM rationale is not a causal explanation; and the detector classifies substantive
interventions — it does not measure policy effectiveness or justice.